<img src="https://iteso.mx/documents/27014/202031/Logo-ITESO-MinimoH.png"
     align="right"
     width="300"/>

# **AttGAN on CelebA**
**Dataset:** CelebA (torchvision) · **Framework:** PyTorch · **Platform:** Google Colab
> He et al. (2019) — *AttGAN: Facial Attribute Editing by Only Changing What You Want*

- Aissa Berenice González Fosado
- Daniela de la Torre Gallo
- Clara Paola Aguilar Casillas

**Experiments**

| Experiment | λ_rec | λ_cls_G | Purpose |
|---|---|---|---|
| `exp1_baseline` | 100 | 1 | Paper defaults — reference |
| `exp2_high_rec` | 200 | 1 | Stronger identity preservation |
| `exp3_strong_attr` | 50 | 5 | Sharper attribute edits |


## *Clone repo & install*

In [1]:
!git clone https://github.com/AissaBerenice/GAN_Project1-DL.git 2>/dev/null || echo 'Repo already cloned'
%cd GAN_Project1-DL
!pip install -q -r requirements.txt

print('Setup complete')

/content/GAN_Project1-DL
Setup complete



## *Verify GPU*

In [2]:
import torch, sys
print(f'Python  : {sys.version.split()[0]}')
print(f'PyTorch : {torch.__version__}')
assert torch.cuda.is_available(), 'No GPU! Go to Runtime > Change Runtime Type > T4 GPU'
print(f'GPU     : {torch.cuda.get_device_name(0)}')
print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

Python  : 3.12.13
PyTorch : 2.10.0+cu128
GPU     : NVIDIA H100 80GB HBM3
VRAM    : 85.0 GB


## *Select experiment*


In [3]:
import importlib, torch

# ── CHANGE THIS ─────────────────────────────────────────────────────
EXPERIMENT = 'exp1_baseline'
# Options: 'exp1_baseline' | 'exp2_high_rec' | 'exp3_strong_attr'
# ────────────────────────────────────────────────────────────────────

_MAP = {
    'exp1_baseline':    ('experiments.exp1_baseline', 'Exp1Config'),
    'exp2_high_rec':    ('experiments.exp2_high_rec', 'Exp2Config'),
    'exp3_strong_attr': ('experiments.exp3_low_rec',  'Exp3Config'),
}
mod_path, cls_name = _MAP[EXPERIMENT]
cfg = getattr(importlib.import_module(mod_path), cls_name)()
device = torch.device('cuda')

print(f'Experiment   : {cfg.EXPERIMENT_NAME}')
print(f'lambda_rec   : {cfg.LAMBDA_REC}')
print(f'lambda_cls_D : {cfg.LAMBDA_CLS_D}')
print(f'lambda_cls_G : {cfg.LAMBDA_CLS_G}')
print(f'Epochs       : {cfg.N_EPOCHS}')
print(f'Batch size   : {cfg.BATCH_SIZE}')
if hasattr(cfg, 'DESCRIPTION'): print(f'Description  : {cfg.DESCRIPTION}')

Experiment   : exp1_baseline
lambda_rec   : 100.0
lambda_cls_D : 10.0
lambda_cls_G : 1.0
Epochs       : 10
Batch size   : 32
Description  : Baseline — paper default lambda values. Balanced reconstruction vs attribute editing.



## *Load CelebA*
torchvision downloads automatically (~1.4 GB, first run only).


In [5]:
#from src.dataset import get_loaders

#train_loader, test_loader = get_loaders(cfg)

#imgs, attrs = next(iter(train_loader))
#print(f'Image batch : {imgs.shape}  (loaded at 128x128, downsampled to 64x64 in training)')
#print(f'Pixel range : [{imgs.min():.2f}, {imgs.max():.2f}]')


## *Build models*
```
image  ->  Encoder (5x Conv2d stride-2)  ->  z (512x4x4)
                                                  |
                          target_attrs -----------+ (tiled, concat)
                                                  |
                                   Generator (5x ConvTranspose2d) -> fake_image
                                                                           |
                                   Discriminator  <------------------------+
                                     |- adv_head  MSELoss (LSGAN)
                                     |- cls_head  BCEWithLogitsLoss (13 attrs)
```

In [12]:
from src.models import build_models
enc, gen, dis = build_models(cfg, device)

[models] Encoder=6.95M  Generator=7.06M  Discriminator=7.07M



## *Train*
| Loss | Function | Weight | Purpose |
|---|---|---|---|
| L_adv | MSELoss (LSGAN) | — | Realism |
| L_cls | BCEWithLogitsLoss | λ_cls_D / λ_cls_G | Correct attributes |
| L_rec | L1Loss | λ_rec | Identity preservation |

Samples + checkpoints saved every 5 epochs to `results/<experiment>/`.

In [13]:
from src.trainer import Trainer

trainer = Trainer(enc, gen, dis, train_loader, test_loader, cfg, device)

# To resume from checkpoint:
# g_losses, d_losses = trainer.train(resume_path='checkpoints/exp1_baseline/ckpt_epoch010.pt')

g_losses, d_losses = trainer.train()


[trainer] exp1_baseline  epochs=10  lambda_rec=100.0  lambda_cls_G=1.0


  batches:   4%|▍         | 211/5086 [00:05<01:27, 55.96it/s]

    step  200  G=25.8504  D=3.4789


  batches:   8%|▊         | 409/5086 [00:08<01:24, 55.17it/s]

    step  400  G=18.4801  D=2.5024


...

  batches:  99%|█████████▊| 5010/5086 [01:30<00:01, 53.97it/s]

    step 5000  G=6.1388  D=1.1234


Epoch [ 10/10]  G=6.2648  D=0.9355
[utils] Saved -> /content/GAN_Project1-DL/results/exp1_baseline/samples_epoch010.png
[utils] Checkpoint -> /content/GAN_Project1-DL/checkpoints/exp1_baseline/ckpt_epoch010.pt

[trainer] Training complete.

[metrics] Building Inception extractor...
Downloading: "https://download.pytorch.org/models/inception_v3_google-0cc3c7bd.pth" to /root/.cache/torch/hub/checkpoints/inception_v3_google-0cc3c7bd.pth


100%|██████████| 104M/104M [00:00<00:00, 214MB/s] 


[metrics] Extracting features (512 images each)...
[metrics] FID   = 58.4652
[metrics] DACID = 3.8787
[trainer] Metrics -> /content/GAN_Project1-DL/results/exp1_baseline/metrics.json


## *Loss curves*

In [14]:
from src.utils import plot_losses
plot_losses(g_losses, d_losses, cfg)

[utils] Saved -> /content/GAN_Project1-DL/results/exp1_baseline/loss_curves.png


## *Attribute manipulation demo*
Each row = one test image. Each column = one attribute toggled independently.  
The model should change **only** the requested attribute.

In [15]:
from src.utils import attribute_demo
attribute_demo(enc, gen, test_loader, cfg, n_imgs=4)

[utils] Saved -> /content/GAN_Project1-DL/results/exp1_baseline/attribute_demo.png



## *Attribute accuracy & reconstruction L1*

In [16]:
from src.utils import evaluate_attribute_accuracy, evaluate_reconstruction

acc = evaluate_attribute_accuracy(enc, gen, dis, test_loader, cfg)
rec = evaluate_reconstruction(enc, gen, test_loader, cfg)

print(f'Attribute accuracy : {acc:.1f}%')
print(f'Reconstruction L1  : {rec:.4f}')


[eval] Attribute accuracy on generated images:
  Bald                    98.6%  ███████████████████
  Bangs                   95.3%  ███████████████████
  Black_Hair              92.0%  ██████████████████
  Blond_Hair              95.5%  ███████████████████
  Brown_Hair              91.4%  ██████████████████
  Bushy_Eyebrows          98.0%  ███████████████████
  Eyeglasses              99.5%  ███████████████████
  Male                    98.8%  ███████████████████
  Mouth_Slightly_Open     99.1%  ███████████████████
  Mustache                99.7%  ███████████████████
  No_Beard                98.3%  ███████████████████
  Pale_Skin               99.1%  ███████████████████
  Young                   95.6%  ███████████████████

  Overall: 97.0%
[eval] Saved -> /content/GAN_Project1-DL/results/exp1_baseline/attr_accuracy.png


/content/GAN_Project1-DL/src/utils.py:197: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(cfg.ATTRS, rotation=45, ha="right", fontsize=8)



[eval] Reconstruction L1: 0.0515  (lower = better)
Attribute accuracy : 97.0%
Reconstruction L1  : 0.0515



## *FID & DACID*
**FID** — Frechet Inception Distance (Heusel et al. 2017). Lower = better.  
**DACID** — custom metric: L2 of mean Inception features. Lower = better.


In [17]:
# cfg.COMPUTE_METRICS = False  # uncomment to skip

if cfg.COMPUTE_METRICS:
    from src.metrics import compute_metrics
    scores = compute_metrics(enc, gen, test_loader, cfg, device)
    print(f"FID   : {scores['fid']}")
    print(f"DACID : {scores['dacid']}")
else:
    scores = {}
    print('Metrics skipped.')


[metrics] Building Inception extractor...
[metrics] Extracting features (512 images each)...
[metrics] FID   = 61.9179
[metrics] DACID = 4.0426
FID   : 61.9179
DACID : 4.0426



## *Save metrics.json*
**Run this at the end of every experiment.**  


In [18]:
import json

payload = {
    'experiment':   cfg.EXPERIMENT_NAME,
    'model':        'AttGAN',
    'lambda_rec':   cfg.LAMBDA_REC,
    'lambda_cls_d': cfg.LAMBDA_CLS_D,
    'lambda_cls_g': cfg.LAMBDA_CLS_G,
    'fid':          scores.get('fid'),
    'dacid':        scores.get('dacid'),
    'attr_acc':     round(float(acc), 2) if 'acc' in dir() else None,
    'rec_l1':       round(float(rec), 4) if 'rec' in dir() else None,
    'g_losses':     g_losses,
    'd_losses':     d_losses,
}
out = cfg.RESULTS_DIR / 'metrics.json'
with open(out, 'w') as f:
    json.dump(payload, f, indent=2)
print(f'Saved -> {out}')
for k in ['lambda_rec','lambda_cls_g','fid','dacid','attr_acc','rec_l1']:
    print(f'  {k:<16}: {payload[k]}')

Saved -> /content/GAN_Project1-DL/results/exp1_baseline/metrics.json
  lambda_rec      : 100.0
  lambda_cls_g    : 1.0
  fid             : 61.9179
  dacid           : 4.0426
  attr_acc        : 96.98
  rec_l1          : 0.0515



## *Download results ZIP*

In [19]:
import shutil
from google.colab import files
zip_name = f'{cfg.EXPERIMENT_NAME}_results'
shutil.make_archive(zip_name, 'zip', cfg.RESULTS_DIR)
files.download(f'{zip_name}.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


## *Compare all experiments*
** after all three experiments are done.**

In [ ]:
import importlib.util
spec = importlib.util.spec_from_file_location('export_results', 'export_results.py')
mod  = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)
mod.main()

## *Fallback*
If the CelebA quota exceeded

Use one of these if Cell 3 raises `FileURLRetrievalError: Too many users`.


In [6]:
# 1. Go to kaggle.com > Account > Create New Token  (downloads kaggle.json)
# 2. Upload kaggle.json via the Colab Files panel (left sidebar)
# 3. Uncomment and run:

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!pip install -q kaggle
!kaggle datasets download -d jessicali9530/celeba-dataset -p data/
!unzip -q data/celeba-dataset.zip -d data/

# Then re-run Cell 3
print('Uncomment above lines after uploading kaggle.json')

cp: cannot stat 'kaggle.json': No such file or directory
Dataset URL: https://www.kaggle.com/datasets/jessicali9530/celeba-dataset
License(s): other
100% 1.33G/1.33G [00:04<00:00, 337MB/s]

Uncomment above lines after uploading kaggle.json


In [7]:
!pip install -q pandas

In [8]:
import torch
import pandas as pd
from PIL import Image
from pathlib import Path
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms as T

_SPLIT_MAP = {"train": 0, "valid": 1, "test": 2}

class KaggleCelebADataset(Dataset):
    def __init__(self, data_dir, split, attr_names, img_size):
        data_dir = Path(data_dir)

        # Find the image folder — handle both possible locations
        for candidate in [
            data_dir / "img_align_celeba",
            data_dir / "celeba" / "img_align_celeba",
        ]:
            if candidate.exists():
                self._img_dir = candidate
                base = candidate.parent
                break
        else:
            raise FileNotFoundError(
                "Cannot find img_align_celeba/ folder inside data/. "
                "Run the unzip cell first."
            )

        # Find CSV files
        for csv_name in ["list_eval_partition.csv", "list_eval_partition.txt"]:
            p = base / csv_name
            if p.exists():
                part_df = pd.read_csv(p)
                break
        else:
            raise FileNotFoundError("Cannot find list_eval_partition.csv")

        for csv_name in ["list_attr_celeba.csv", "list_attr_celeba.txt"]:
            p = base / csv_name
            if p.exists():
                attr_df = pd.read_csv(p)
                break
        else:
            raise FileNotFoundError("Cannot find list_attr_celeba.csv")

        # Standardise column names
        part_df.columns = part_df.columns.str.strip()
        attr_df.columns  = attr_df.columns.str.strip()

        # Rename first column to 'image_id' if needed
        if part_df.columns[0] != 'image_id':
            part_df = part_df.rename(columns={part_df.columns[0]: 'image_id'})
        if attr_df.columns[0] != 'image_id':
            attr_df = attr_df.rename(columns={attr_df.columns[0]: 'image_id'})

        # Rename partition column if needed
        pcol = [c for c in part_df.columns if c != 'image_id'][0]
        part_df = part_df.rename(columns={pcol: 'partition'})

        merged   = part_df.merge(attr_df, on='image_id')
        split_df = merged[merged['partition'] == _SPLIT_MAP[split]].reset_index(drop=True)

        self._filenames  = split_df['image_id'].tolist()
        # Kaggle attrs are already {-1,+1}; torchvision uses {0,1} — handle both
        raw = split_df[attr_names].values.astype('float32')
        # If all values are 0 or 1, convert to {-1,+1}
        if raw.max() <= 1.0 and raw.min() >= 0.0:
            raw = raw * 2 - 1
        self._attrs = raw

        self._transform = T.Compose([
            T.CenterCrop(178),
            T.Resize(img_size),
            T.RandomHorizontalFlip(),
            T.ToTensor(),
            T.Normalize([0.5]*3, [0.5]*3),
        ])

    def __len__(self): return len(self._filenames)

    def __getitem__(self, i):
        img = Image.open(self._img_dir / self._filenames[i]).convert('RGB')
        return self._transform(img), torch.from_numpy(self._attrs[i])


def get_loaders(cfg):
    kw = dict(data_dir=cfg.DATA_DIR, attr_names=cfg.ATTRS, img_size=cfg.IMG_SIZE)
    train_ds = KaggleCelebADataset(split='train', **kw)
    test_ds  = KaggleCelebADataset(split='test',  **kw)
    lkw = dict(num_workers=2, pin_memory=True, drop_last=True)
    train_loader = DataLoader(train_ds, batch_size=cfg.BATCH_SIZE, shuffle=True,  **lkw)
    test_loader  = DataLoader(test_ds,  batch_size=cfg.BATCH_SIZE, shuffle=False, **lkw)
    print(f"[dataset] train={len(train_ds):,}  test={len(test_ds):,}")
    return train_loader, test_loader

# Make this the active loader for this session
import src.dataset as _ds_module
import src.simple_gan as _sg_module
_ds_module.get_loaders = get_loaders
print("Dataset loader patched for Kaggle files.")

Dataset loader patched for Kaggle files.


In [9]:
from src.dataset import get_loaders

train_loader, test_loader = get_loaders(cfg)

[dataset] train=162,770  test=19,962


In [10]:
from pathlib import Path

# Point the loader to the correct nested folder
correct_path = Path('data/img_align_celeba/img_align_celeba')
train_loader.dataset._img_dir = correct_path
test_loader.dataset._img_dir  = correct_path

print(f"img_dir set to: {correct_path}")
print(f"Exists: {correct_path.exists()}")
print(f"Sample files: {list(correct_path.iterdir())[:3]}")

img_dir set to: data/img_align_celeba/img_align_celeba
Exists: True
Sample files: [PosixPath('data/img_align_celeba/img_align_celeba/158933.jpg'), PosixPath('data/img_align_celeba/img_align_celeba/075701.jpg'), PosixPath('data/img_align_celeba/img_align_celeba/198830.jpg')]


In [11]:
imgs, attrs = next(iter(train_loader))
print(f'Image batch : {imgs.shape}')
print(f'Pixel range : [{imgs.min():.2f}, {imgs.max():.2f}]')

Image batch : torch.Size([32, 3, 128, 128])
Pixel range : [-1.00, 1.00]


In [ ]:
from pathlib import Path
import torch
import pandas as pd
from PIL import Image
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms as T

_SPLIT_MAP = {"train": 0, "valid": 1, "test": 2}

class KaggleCelebADataset(Dataset):
    def __init__(self, img_dir, base_dir, split, attr_names, img_size):
        self._img_dir = Path(img_dir)

        part_df = pd.read_csv(Path(base_dir) / 'list_eval_partition.csv')
        attr_df  = pd.read_csv(Path(base_dir) / 'list_attr_celeba.csv')
        part_df.columns = part_df.columns.str.strip()
        attr_df.columns  = attr_df.columns.str.strip()

        if part_df.columns[0] != 'image_id':
            part_df = part_df.rename(columns={part_df.columns[0]: 'image_id'})
        if attr_df.columns[0] != 'image_id':
            attr_df = attr_df.rename(columns={attr_df.columns[0]: 'image_id'})

        pcol = [c for c in part_df.columns if c != 'image_id'][0]
        part_df = part_df.rename(columns={pcol: 'partition'})

        merged   = part_df.merge(attr_df, on='image_id')
        split_df = merged[merged['partition'] == _SPLIT_MAP[split]].reset_index(drop=True)

        self._filenames = split_df['image_id'].tolist()
        raw = split_df[attr_names].values.astype('float32')
        if raw.max() <= 1.0 and raw.min() >= 0.0:
            raw = raw * 2 - 1
        self._attrs = raw

        self._transform = T.Compose([
            T.CenterCrop(178),
            T.Resize(img_size),
            T.RandomHorizontalFlip(),
            T.ToTensor(),
            T.Normalize([0.5]*3, [0.5]*3),
        ])

    def __len__(self): return len(self._filenames)

    def __getitem__(self, i):
        img = Image.open(self._img_dir / self._filenames[i]).convert('RGB')
        return self._transform(img), torch.from_numpy(self._attrs[i])


# Hard-coded correct paths based on your folder structure
IMG_DIR  = 'data/img_align_celeba/img_align_celeba'
BASE_DIR = 'data'

kw = dict(img_dir=IMG_DIR, base_dir=BASE_DIR,
          attr_names=cfg.ATTRS, img_size=cfg.IMG_SIZE)

train_ds = KaggleCelebADataset(split='train', **kw)
test_ds  = KaggleCelebADataset(split='test',  **kw)

lkw = dict(num_workers=2, pin_memory=True, drop_last=True)
train_loader = DataLoader(train_ds, batch_size=cfg.BATCH_SIZE, shuffle=True,  **lkw)
test_loader  = DataLoader(test_ds,  batch_size=cfg.BATCH_SIZE, shuffle=False, **lkw)

print(f"train={len(train_ds):,}  test={len(test_ds):,}")

# Verify
imgs, attrs = next(iter(train_loader))
print(f"Batch: {imgs.shape}  range=[{imgs.min():.2f}, {imgs.max():.2f}]")
print("Loaders ready — run the train cell now.")

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# import os; os.makedirs('data', exist_ok=True)
# !ln -sfn '/content/drive/MyDrive/YOUR_CELEBA_FOLDER' data/celeba
# # Then re-run Cell 4
print('Uncomment above and set YOUR_CELEBA_FOLDER')